# T5-large training with LoRA
Reconstructed from your uploaded notebook. Uses your original question/context format and JSON splits. This trains a **new** adapter; it does not reproduce the exact old weights.

In [ ]:
# Install once if needed:
# %pip install -U transformers datasets peft accelerate sentencepiece

## 1. Load the original datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": "dataset/train.json",
    "val": "dataset/val.json",
    "test": "dataset/test.json"
})
print(ds)
print(ds["train"][0])

## 2. Prepare question/context inputs and answers

In [ ]:
def process(dataset):
    input_data = [f"question: {q} context: {c}" for q, c in zip(dataset["question"], dataset["context"])]
    return {"input": input_data, "target": dataset["answer"]}

processed_data = ds.map(process, batched=True, batch_size=100,
                        remove_columns=ds["train"].column_names)
print(processed_data["train"][0])

## 3. Tokenize for T5-large
The old notebook used `t5-base` for tokenization, then `t5-small` for training. Here both use `t5-large`.

In [ ]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-large")

def tokenizator(dataset):
    tokenized_inputs = tokenizer(dataset["input"], max_length=512,
                                 padding="max_length", truncation=True)
    tokenized_target = tokenizer(dataset["target"], max_length=128,
                                 padding="max_length", truncation=True)
    labels = tokenized_target["input_ids"]
    labels = [[-100 if token == tokenizer.pad_token_id else token for token in row]
              for row in labels]
    return {"input_ids": tokenized_inputs["input_ids"],
            "attention_mask": tokenized_inputs["attention_mask"],
            "labels": labels}

tokenized_data = processed_data.map(tokenizator, batched=True, batch_size=100,
                                    remove_columns=["input", "target"])
print(tokenized_data)

## 4. Load T5-large and add LoRA
LoRA settings below match the configuration recovered from your existing adapter. Standard T5-large uses 32,128 vocabulary entries; the recovered adapter had 32,100. This new training intentionally uses the standard matching T5-large model and tokenizer. Investigate the old tokenizer before attempting to load the recovered adapter into this new model.

In [ ]:
import torch
from transformers import T5ForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

model = T5ForConditionalGeneration.from_pretrained("t5-large")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Train
Batch size 16 was in the old notebook but is likely too large for T5-large on a typical GPU. Start with 1 and gradient accumulation of 16. `fp16` requires compatible CUDA hardware; adjust if needed.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./t5_large_lora_training",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    save_strategy="epoch",
    eval_strategy="epoch",
    per_device_eval_batch_size=1,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["val"]
)

trainer.train()

## 6. Save the new LoRA adapter

In [ ]:
model.save_pretrained("./lora-t5-large-new")
tokenizer.save_pretrained("./lora-t5-large-new")
print("Saved new adapter and tokenizer to ./lora-t5-large-new")

**Note:** If your installed Transformers version reports an unexpected argument for `eval_strategy`, replace it with `evaluation_strategy`. The new checkpoint will not have identical weights to the recovered one.